In [20]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


## Extraction from PDF

In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass, field
import pymupdf


@dataclass
class ExtractorConfig:
    header_max_y: float = 80.0       # running header sits above this y
    body_x_min: float = 150.0        # body column left edge (left-margin notes are at x~91)
    body_x_max: float = 458.0        # body column right edge (right-margin notes are at x~463)
    footnote_font_max: float = 8.5   # footnotes are 8pt, body is 11pt
    marker_font_max: float = 7.0     # superscript footnote numbers are 4-5.5pt
    separator_min_len: int = 30      # footnote separator = long run of dashes
    note_y_tolerance: float = 6.0


@dataclass
class MarginNote:
    y0: float
    text: str


@dataclass
class PageContent:
    pdf_page: int
    printed_page: str | None
    body: str
    margin_notes: list[MarginNote] = field(default_factory=list)
    footnotes: list[tuple[str, str]] = field(default_factory=list)


_SEPARATOR_CHARS = set("–—-_ ")
_PAGE_TOKEN = re.compile(r"\d{1,3}|[ivxlc]{1,6}", re.I)
_FOOTER_ROMAN = re.compile(r"^\(([ivxlc]+)\)$", re.I)


class PDFExtractor:
    def __init__(self, config: ExtractorConfig | None = None):
        self.cfg = config or ExtractorConfig()

    def extract(self, path: str) -> list[PageContent]:
        doc = pymupdf.open(path)
        pages = [self._extract_page(p, i + 1) for i, p in enumerate(doc)]
        doc.close()
        return pages

    def _extract_page(self, page, pdf_page: int) -> PageContent:
        c = self.cfg
        blocks = [b for b in page.get_text("dict")["blocks"] if b["type"] == 0]
        sep_y = self._find_separator_y(blocks)

        printed_page = None
        rows = []             # (baseline_y, x0, text)
        margin_spans = []     # (side, baseline, text), one per visual margin line
        footnote_lines = []   # each line = [(font_size, text), ...]

        for b in blocks:
            # header: running title + page number
            if b["bbox"][1] < c.header_max_y:
                big = " ".join(s["text"] for ln in b["lines"] for s in ln["spans"]
                               if s["size"] > c.marker_font_max)
                big = re.sub(r"\s+", " ", big).strip()
                rest = re.sub(r"CONSTITUTION OF PAKISTAN", "", big, flags=re.I).strip()
                if "CONSTITUTION OF PAKISTAN" in big.upper() or _PAGE_TOKEN.fullmatch(rest):
                    if _PAGE_TOKEN.fullmatch(rest):
                        printed_page = rest
                    continue

            for ln in b["lines"]:
                # classify each span on its own: a margin title can share a row with body text
                body_sp, margin_sp = [], []
                for s_ in ln["spans"]:
                    x = s_["bbox"][0]
                    if s_["text"].strip() == "":
                        body_sp.append(s_)
                    elif x < c.body_x_min or x > c.body_x_max:
                        margin_sp.append(s_)
                    else:
                        body_sp.append(s_)

                m_txt = "".join(s_["text"] for s_ in margin_sp
                                if s_["size"] > c.marker_font_max).strip()
                if m_txt:
                    side = "L" if margin_sp[0]["bbox"][0] < c.body_x_min else "R"
                    base = max(margin_sp, key=lambda q: q["size"])["origin"][1]
                    margin_spans.append((side, base, m_txt))

                if not any(s_["text"].strip() for s_ in body_sp):
                    continue
                if sep_y is not None and ln["bbox"][1] >= sep_y:
                    footnote_lines.append([(s_["size"], s_["text"]) for s_ in body_sp])
                else:
                    txt = "".join(s_["text"] for s_ in body_sp if s_["size"] > c.marker_font_max)
                    if txt.strip():
                        base = max(body_sp, key=lambda q: q["size"])["origin"][1]
                        rows.append((base, body_sp[0]["bbox"][0], txt.rstrip()))

        margin_notes = self._group_margin_lines(margin_spans)
        footnotes = self._build_footnotes(footnote_lines)
        rows = self._merge_same_row(rows)

        if printed_page is None:  # roman numeral footer like "(xv)"
            for _, _, text in rows:
                m = _FOOTER_ROMAN.match(text.strip())
                if m:
                    printed_page = m.group(1)
                    break

        margin_notes.sort(key=lambda n: n.y0)
        body = self._assemble_body(rows, margin_notes)
        return PageContent(pdf_page, printed_page, body, margin_notes, footnotes)
    
    @staticmethod
    def _group_margin_lines(lines) -> list[MarginNote]:
        """Join consecutive margin lines (same side, <=12pt apart) into one note."""
        notes = []
        for side in ("L", "R"):
            cur, last = None, None
            for _, base, text in sorted((l for l in lines if l[0] == side), key=lambda l: l[1]):
                if cur is not None and base - last <= 12:
                    cur[1] += " " + text
                else:
                    if cur:
                        notes.append(cur)
                    cur = [base, text]
                last = base
            if cur:
                notes.append(cur)
        return [MarginNote(b, re.sub(r"\s+", " ", t).strip())
                for b, t in notes if re.search(r"[A-Za-z]", t)]
    
    @staticmethod
    def _baseline(line) -> float:
        # baseline of the largest span: stable even when a superscript changes the bbox
        return max(line["spans"], key=lambda s: s["size"])["origin"][1]

    @staticmethod
    def _merge_same_row(rows):
        rows = sorted(rows, key=lambda r: (round(r[0], 0), r[1]))
        merged = []
        for base, x0, text in rows:
            if merged and abs(merged[-1][0] - base) < 2.0:
                pb, px, pt = merged[-1]
                merged[-1] = (pb, px, pt + " " + text.strip())
            else:
                merged.append((base, x0, text))
        return merged

    def _assemble_body(self, rows, notes) -> str:
        if not rows:
            return ""
        gaps = sorted(rows[i][0] - rows[i - 1][0] for i in range(1, len(rows)))
        pitch = gaps[len(gaps) // 4] if gaps else 13.0
        para_gap = max(pitch * 1.35, 1.0)

        # attach each margin note to the nearest row at/above it
        note_at = {}
        for note in notes:
            idx = 0
            for i, (base, _, _) in enumerate(rows):
                if base <= note.y0 + self.cfg.note_y_tolerance:
                    idx = i
            note_at.setdefault(idx, []).append(note.text)

        out = []
        for i, (base, _, text) in enumerate(rows):
            if i in note_at:
                out.append("\n\n" if out else "")
                out.append("\n".join(f"[[NOTE: {t}]]" for t in note_at[i]) + "\n")
            elif out:
                out.append("\n\n" if base - rows[i - 1][0] > para_gap else "\n")
            out.append(text)
        return "".join(out)

    def _find_separator_y(self, blocks):
        c = self.cfg
        for b in blocks:
            for ln in b["lines"]:
                text = "".join(s["text"] for s in ln["spans"]).strip()
                size = max((s["size"] for s in ln["spans"]), default=99)
                if (len(text) >= c.separator_min_len and set(text) <= _SEPARATOR_CHARS
                        and size <= c.footnote_font_max):
                    return ln["bbox"][1]
        return None

    def _build_footnotes(self, lines):
        c = self.cfg
        notes = []  # [number, text]
        for spans in lines:
            buf = ""
            for size, txt in spans:
                if size <= c.marker_font_max and txt.strip():
                    if buf.strip() and notes:
                        notes[-1][1] += " " + buf.strip()
                    buf = ""
                    notes.append([txt.strip().rstrip("."), ""])
                else:
                    buf += txt
            if buf.strip() and notes:
                notes[-1][1] += " " + buf.strip()
        return [(n, re.sub(r"\s+", " ", t).strip()) for n, t in notes if t.strip()]

In [ ]:
PDF_PATH = "constitution_of_Pakistan.pdf"   # adjust to your path

pages = PDFExtractor().extract(PDF_PATH)
print(len(pages), "pages")                  # expect 252
print(pages[40].printed_page)               # expect "19"
print(pages[40].body[-600:])                # should show [[NOTE: ...]] before "38."
print(pages[27].footnotes[:2])              # amendment footnotes

252 pages
19
hands of a
few to the detriment of general interest and by
ensuring equitable adjustment of rights between
employers and employees, and landlords and tenants;

        (b)    provide for all citizens, within the available resources
of the country, facilities for work and adequate
livelihood with reasonable rest and leisure;

        (c)    provide for all persons employed in the service of
Pakistan or otherwise, social security by compulsory
social insurance or other means;

        (d)    provide basic necessities of life, such as food,
clothing, housing, education and medical relief, for all
[('1', 'Subs. by P. O. No. 14 of 1985, Art. 2 and Sch., for ‘‘basic’’.'), ('2', 'Subs. by the Constitution (Eighteenth Amdt.) Act, 2010 (10 of 2010), s. 4, for “clause (1)”.')]


In [ ]:
body = pages[40].body
i = body.index("[[NOTE:")
print(body[i-150 : i+250])

alise the Government administration so as to
facilitate expeditious disposal of its business to meet
the convenience and requirements of the public.

[[NOTE: Promotion of social and economic well- being of the people.]]
38.   The State shall––

        (a)    secure the well-being of the people, irrespective of
sex, caste, creed or race, by raising their standard of
living, by preventing the conce


## Text Cleaning

In [ ]:
import re
from bisect import bisect_right
from dataclasses import dataclass


@dataclass
class CleanedDocument:
    text: str
    page_starts: list         # (char_offset, pdf_page, printed_page)
    footnotes_by_page: dict   # pdf_page -> ["1. Subs. by ...", ...]

    def page_at(self, offset: int):
        i = bisect_right([s[0] for s in self.page_starts], offset) - 1
        _, pdf_page, printed = self.page_starts[max(i, 0)]
        return pdf_page, printed


class TextCleaner:
    TERMINAL = ".;:,–—?!"
    LABEL = re.compile(r"^(\(\w{1,5}\)|\d{1,3}[A-Z]{0,2}\.)(\s|$)")
    HEADING = re.compile(r"^(PART [IVXLC]+\b|CHAPTER \d+[A-Z]?\b)")
    XREF_FOOTNOTE = re.compile(r"^See\b.*\b(footnote|supra)\b", re.I)
    NOTE = re.compile(r"\[\[NOTE: (.*?)\]\]")

    def __init__(self, first_content_page: int = 23):
        self.first_content_page = first_content_page  # skips cover, preface, table of contents

    def clean(self, pages) -> CleanedDocument:
        text, starts, footnotes = "", [], {}
        for p in pages:
            if p.pdf_page < self.first_content_page:
                continue
            body = self._clean_page(p.body)
            if not body:
                continue
            if text:
                text += self._joiner(text, body)
            starts.append((len(text), p.pdf_page, p.printed_page))
            text += body
            notes = [f"{n}. {t}" for n, t in p.footnotes if not self.XREF_FOOTNOTE.match(t)]
            if notes:
                footnotes[p.pdf_page] = notes
        return CleanedDocument(text, starts, footnotes)

    # ---------------------------------------------------------------- per page
    def _clean_page(self, body: str) -> str:
        if not body.strip():
            return ""
        # 1) protect margin notes; each becomes its own paragraph
        stash = []
        def _stash(m):
            stash.append(self._tidy(m.group(1)))
            return f"\x01{len(stash) - 1}\x02\n\n"
        t = self.NOTE.sub(_stash, body)

        # 2) omitted provisions ("46. *", "(2) * * *") -> stub; other asterisks = repealed words
        t = re.sub(r"(?m)^(\(\w{1,5}\)|\d{1,3}[A-Z]{0,2}\.)[^\S\n]*[\*\s]+$", r"\1 Omitted.", t)
        t = re.sub(r"(?m)^[\*\s]+$", "", t)
        t = re.sub(r"[^\S\n]*(?:\*[^\S\n]*)+", " ", t)

        # 3) square brackets mark amended text: drop them, tidy spacing
        t = self._tidy(t)

        # 4) rebuild paragraphs from wrapped lines
        paragraphs = []
        for para in re.split(r"\n\s*\n", t):
            lines = [re.sub(r"[^\S\n]+", " ", ln).strip() for ln in para.split("\n")]
            lines = [self._norm_heading(ln) for ln in lines if ln]
            for group in self._split_at_headings(lines):
                paragraphs.append(self._join_lines(group))
        paragraphs = self._fix_headings_and_notes(paragraphs)
        out = "\n\n".join(paragraphs)

        # 5) restore margin notes
        return re.sub(r"\x01(\d+)\x02",
                      lambda m: f"[[NOTE: {stash[int(m.group(1))]}]]", out).strip()

    @staticmethod
    def _tidy(s: str) -> str:
        s = s.replace("[", " ").replace("]", " ")
        s = s.replace("‘‘", "“").replace("’’", "”")
        s = re.sub(r"(?<=[A-Za-z])- (?=[a-z])", "-", s)       # "well- being" -> "well-being"
        s = re.sub(r"[^\S\n]+([;,.:)])", r"\1", s)             # "Minister ;" -> "Minister;"
        s = re.sub(r"(?<!\.)\.\.(?!\.)", ".", s)               # stray ".."
        s = re.sub(r"“[^\S\n]+", "“", s)
        s = re.sub(r"[^\S\n]+”", "”", s)
        return re.sub(r"[^\S\n]+", " ", s).strip(" ") if "\n" not in s else s

    def _fix_headings_and_notes(self, paras):
        """PART X + title on one line; keep margin notes by their article, not above a heading."""
        out, i = [], 0
        while i < len(paras):
            p = paras[i]
            m = re.match(r"^PART ([IVXLC]+)\b[^\S\n]*[–—-]*[^\S\n]*(.*)$", p)
            if m and "\n" not in p:
                title, held = m.group(2).strip(), []
                if not title:
                    k = i + 1
                    while k < len(paras) and paras[k].startswith("\x01"):  # notes before the title
                        held.append(paras[k]); k += 1
                    if k < len(paras):
                        nxt = paras[k]
                        if (len(nxt) <= 80 and not nxt.endswith(".") and not nxt.startswith("(")
                                and not self.HEADING.match(nxt) and not re.match(r"^\d", nxt)):
                            title, i = nxt, k
                        else:
                            held = []
                out.append(f"PART {m.group(1)} – {title}" if title else f"PART {m.group(1)}")
                out.extend(held)
            else:
                out.append(p)
            i += 1
        # a note directly above a PART/CHAPTER heading belongs after the heading(s)
        j = 0
        while j < len(out) - 1:
            if out[j].startswith("\x01") and self.HEADING.match(out[j + 1]):
                out[j], out[j + 1] = out[j + 1], out[j]
                j = max(j - 1, 0)
            else:
                j += 1
        return out

    ARTICLE_LINE = re.compile(r"^\d{1,3}[A-Z]{0,2}\.\s")

    def _split_at_headings(self, lines):
        """A PART/CHAPTER line always starts a new paragraph; so does an article right after a CHAPTER line."""
        groups, cur = [], []
        for ln in lines:
            if cur and (self.HEADING.match(ln)
                        or (cur[-1].startswith("CHAPTER") and self.ARTICLE_LINE.match(ln))):
                groups.append(cur)
                cur = []
            cur.append(ln)
        if cur:
            groups.append(cur)
        return groups
    
    @staticmethod
    def _norm_heading(line: str) -> str:
        m = re.match(r"^CHAPTER\s+(\d+[A-Z]?)\s*\.?\s*[–—-]+\s*(.*)$", line)
        return f"CHAPTER {m.group(1)} – {m.group(2)}".strip() if m else line

    def _join_lines(self, lines):
        out = lines[0]
        for ln in lines[1:]:
            if re.search(r"\w-$", out):                              # wrap inside hyphenated word
                out += ln
            elif self.LABEL.match(ln) and out[-1] in self.TERMINAL:  # new clause starts here
                out += "\n" + ln
            else:
                out += " " + ln
        return out

    def _joiner(self, prev: str, nxt: str) -> str:
        """Glue page N to page N+1: continue the sentence, or start a new paragraph."""
        ends_sentence = prev.rstrip()[-1:] in self.TERMINAL
        starts_new = nxt.startswith("[[NOTE:") or self.HEADING.match(nxt)
        return "\n\n" if (ends_sentence or starts_new) else " "

In [ ]:
pages = PDFExtractor().extract(PDF_PATH)
doc = TextCleaner().clean(pages)
t = doc.text

print(len(t), "chars;", len(doc.page_starts), "pages;", len(doc.footnotes_by_page), "pages with footnotes")

i = t.index("97. Subject to the Constitution")
print(t[i-120:i+200])                       # note should sit directly above "97."

i = t.index("PART IV")
print(t[i:i+200])                          # PART IV – Provinces / CHAPTER 1 / note / 101.

nums = {int(m.group(1)) for m in re.finditer(r"(?m)^(\d{1,3})[A-Z]{0,2}\. ", t)}
print("missing articles:", sorted(set(range(1, 281)) - nums))   # expect []

378487 chars; 222 pages; 159 pages with footnotes
by Act 23 of 1977, s.2, and come into force on 16th May, 1977.

[[NOTE: Extent of executive authority of Federation.]]

97. Subject to the Constitution, the executive authority of the Federation shall extend to the matters with respect to which Majlis-e-Shoora (Parliament) has power to make laws, including exercise of 
PART IV – Provinces

CHAPTER 1 – THE GOVERNORS

[[NOTE: Appointment of Governor.]]

101. (1) There shall be a Governor for each Province, who shall be appointed by the President on the advice of the P
missing articles: []


In [ ]:
print(hasattr(PDFExtractor, "_group_margin_lines"))   # must be True
print(ExtractorConfig().note_y_tolerance)             # must be 6.0

True
6.0


## ArticleChunker

In [24]:
import re
from dataclasses import dataclass


@dataclass
class Chunk:
    id: str
    text: str        # what gets embedded: context header + body
    metadata: dict   # flat values only (str/int/float/bool): Chroma rejects lists and None


class ArticleChunker:
    DOC = "Constitution of Pakistan"
    ART_START = re.compile(r"^(\d{1,3})([A-Z]{0,2})\.(?=[\s–—-])")
    PART = re.compile(r"^PART ([IVXLC]+)\b")
    CHAPTER = re.compile(r"^CHAPTER \d+[A-Z]?\b")
    SEPARATOR = re.compile(r"^[_–—\-\s]{3,}$")
    ANNEX_SPLIT = re.compile(r"(?m)^_{3,}\s*ANNEX")
    SCHEDULE = re.compile(r"^(FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH) SCHEDULE\b")
    TERMINAL = ".;:,–—?!”)"

    def __init__(self, max_chars: int = 1500):
        self.max_chars = max_chars   # keeps header+body under bge-small's 512-token window

    # ------------------------------------------------------------------ public
    def chunk(self, doc) -> list[Chunk]:
        text = doc.text
        m = self.ANNEX_SPLIT.search(text)
        main, tail, tail_base = (text[:m.start()], text[m.start():], m.start()) if m else (text, "", 0)
        chunks = self._chunk_main(doc, main)
        if tail:
            chunks += self._chunk_tail(doc, tail, tail_base)
        return chunks

    # ----------------------------------------------- preamble + articles 1..280
    def _chunk_main(self, doc, text):
        chunks, buf = [], []                 # buf = preamble paragraphs
        part = chapter = sub = pending = None
        cur = None                           # article under construction
        last_key = None
        in_preamble = True

        paras = list(self._paras(text))
        for i, (off, p) in enumerate(paras):
            nxt = paras[i + 1][1] if i + 1 < len(paras) else ""
            if self.SEPARATOR.match(p):
                continue

            if self.PART.match(p):
                if in_preamble:
                    chunks += self._emit(doc, buf, "preamble", "Preamble", {"article_number": "Preamble"})
                    in_preamble = False
                chunks += self._flush(doc, cur); cur = None
                part, chapter, sub = p, None, None
                continue
            if in_preamble:
                buf.append((off, p)); continue

            if self.CHAPTER.match(p):
                chunks += self._flush(doc, cur); cur = None
                chapter, sub = p.rstrip("."), None
                continue

            if p.startswith("[[NOTE:"):
                note = p[len("[[NOTE: "):-2]
                if self._is_article_start(nxt, last_key) or self.CHAPTER.match(nxt) or self.PART.match(nxt):
                    pending = f"{pending} / {note}" if pending else note
                elif cur is not None and not cur["title"]:
                    cur["title"] = note            # note landed one line late: it titles the open article
                else:
                    pending = f"{pending} / {note}" if pending else note
                continue

            if self._is_article_start(p, last_key):
                chunks += self._flush(doc, cur)
                num, suf = self.ART_START.match(p).groups()
                last_key = (int(num), suf)
                cur = {"num": int(num), "suf": suf, "title": pending, "part": part,
                       "chapter": chapter, "sub": sub, "paras": [(off, p)]}
                pending = None
                continue

            # short unpunctuated paragraph right before a margin note = sub-heading inside a chapter
            if nxt.startswith("[[NOTE:") and len(p) < 120 and p[-1] not in self.TERMINAL and not p.startswith("("):
                chunks += self._flush(doc, cur); cur = None
                sub = p
                continue

            if cur is not None:
                cur["paras"].append((off, p))

        chunks += self._flush(doc, cur)
        return chunks

    def _is_article_start(self, p, last_key) -> bool:
        m = self.ART_START.match(p)
        if not m:
            return False
        key = (int(m.group(1)), m.group(2))
        return last_key is None or (key > last_key and key[0] - last_key[0] <= 10)

    def _flush(self, doc, a):
        if not a:
            return []
        num = f"{a['num']}{a['suf']}"
        extra = {"article_number": num, "article_num": a["num"]}
        # omitted articles carry their title inline: "71. Mediation Committee Omitted by ..."
        mo = re.match(r"^\d{1,3}[A-Z]{0,2}\.\s+(.*?)\.?\s+Omitted\b", a["paras"][0][1])
        if len(a["paras"]) == 1 and mo:
            extra["omitted"] = True
            a["title"] = a["title"] or mo.group(1).strip()
        for k_src, k_dst in (("title", "article_title"), ("part", "part"),
                             ("chapter", "chapter"), ("sub", "subheading")):
            if a[k_src]:
                extra[k_dst] = a[k_src]
        label = f"Article {num}" + (f": {a['title']}" if a["title"] else "")
        ctx = " | ".join(x for x in (self.DOC, a["part"], a["chapter"], a["sub"], label) if x)
        return self._emit(doc, a["paras"], "article", ctx, extra, id_base=f"art-{num}")

    # ------------------------------------------------------ annex + schedules
    def _chunk_tail(self, doc, text, base):
        chunks, section, kind, paras = [], None, None, []

        def close():
            if paras:
                slug = re.sub(r"[^a-z0-9]+", "-", section.lower()).strip("-")
                chunks.extend(self._emit(doc, paras, kind, f"{self.DOC} | {section}",
                                         {"section": section}, id_base=slug))

        for off, p in self._paras(text, base):
            if re.match(r"^_{3,}\s*ANNEX", p):
                close(); section, kind, paras = "Annex (Objectives Resolution)", "annex", []
                continue
            m = self.SCHEDULE.match(p)
            if m:
                close(); section, kind, paras = f"{m.group(1).title()} Schedule", "schedule", []
                paras.append((off, p)); continue
            if self.SEPARATOR.match(p):
                continue
            if section:
                paras.append((off, p))
        close()
        return chunks

    # ------------------------------------------------------ shared machinery
    @staticmethod
    def _paras(text, base=0):
        pos = 0
        for part in text.split("\n\n"):
            s = part.strip()
            if s:
                yield base + pos + (len(part) - len(part.lstrip())), s
            pos += len(part) + 2

    def _emit(self, doc, paras, kind, ctx, extra, id_base=None):
        paras = [(o, re.sub(r"\s*[_–—\-]{4,}\s*$", "", p)) for o, p in paras]   # trailing rule lines
        paras = [(o, p) for o, p in paras if p]
        if not paras:
            return []
        packed = self._pack(paras)
        out = []
        for k, (first_off, last_off, body) in enumerate(packed):
            p0, printed = doc.page_at(first_off)
            p1, _ = doc.page_at(last_off)
            notes, seen = [], set()
            for pg in range(p0, p1 + 1):
                for n in doc.footnotes_by_page.get(pg, []):
                    if n not in seen:
                        seen.add(n); notes.append(n)
            meta = {"doc": self.DOC, "kind": kind, "chunk_index": k, "chunk_count": len(packed),
                    "pdf_page": p0, **extra}
            if printed:
                meta["printed_page"] = printed
            if notes:
                meta["amendment_notes"] = " | ".join(notes)[:2000]
            header = ctx if len(packed) == 1 else f"{ctx} (part {k + 1}/{len(packed)})"
            cid = id_base or kind
            out.append(Chunk(id=cid if len(packed) == 1 else f"{cid}-p{k + 1}",
                             text=f"{header}\n\n{body}", metadata=meta))
        return out

    def _pack(self, paras):
        limit = self.max_chars
        pieces = []
        for off, p in paras:
            pieces += [(off, s) for s in (self._split_long(p, limit) if len(p) > limit else [p])]
        chunks, cur, n = [], [], 0
        for off, s in pieces:
            if cur and n + len(s) + 2 > limit:
                chunks.append(cur); cur, n = [], 0
            cur.append((off, s)); n += len(s) + 2
        if cur:
            chunks.append(cur)
        return [(c[0][0], c[-1][0], "\n\n".join(s for _, s in c)) for c in chunks]

    @staticmethod
    def _split_long(p, limit):
        """Split an oversized clause at sentence/semicolon boundaries, then at spaces as a last resort."""
        parts, cur = [], ""
        for sent in re.split(r"(?<=[.;:])\s+", p):
            while len(sent) > limit:
                cut = sent.rfind(" ", 0, limit)
                cut = cut if cut > 0 else limit
                if cur:
                    parts.append(cur); cur = ""
                parts.append(sent[:cut]); sent = sent[cut:].lstrip()
            if cur and len(cur) + len(sent) + 1 > limit:
                parts.append(cur); cur = sent
            else:
                cur = f"{cur} {sent}".strip()
        if cur:
            parts.append(cur)
        return parts

## vector store + orchestrator

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

class VectorStore:
    """All Chroma-specific code lives here, so swapping to pgvector later means rewriting only this class."""

    def __init__(self, path="./chroma_db", collection="pakistan_constitution",
                 model="BAAI/bge-small-en-v1.5"):
        self.client = chromadb.PersistentClient(path=path)
        self.name = collection
        self.ef = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model, normalize_embeddings=True)
        self.collection = self._open()

    def _open(self):
        return self.client.get_or_create_collection(
            name=self.name, embedding_function=self.ef, metadata={"hnsw:space": "cosine"})

    def reset(self):
        """Drop and recreate, so chunks whose ids changed don't linger after re-chunking."""
        try:
            self.client.delete_collection(self.name)
        except Exception:
            pass
        self.collection = self._open()

    def upsert(self, chunks, batch_size=64):
        for i in range(0, len(chunks), batch_size):
            b = chunks[i:i + batch_size]
            self.collection.upsert(ids=[c.id for c in b],
                                   documents=[c.text for c in b],
                                   metadatas=[c.metadata for c in b])

    def query(self, text, n_results=5, where=None):
        return self.collection.query(query_texts=[text], n_results=n_results, where=where)

    def get_article(self, number):
        return self.collection.get(where={"article_number": str(number)})


class ConstitutionIngestor:
    def __init__(self, extractor, cleaner, chunker, store):
        self.extractor, self.cleaner, self.chunker, self.store = extractor, cleaner, chunker, store

    def run(self, path, reset=False):
        pages = self.extractor.extract(path)
        doc = self.cleaner.clean(pages)
        chunks = self.chunker.chunk(doc)
        if reset:
            self.store.reset()
        self.store.upsert(chunks)
        return chunks

In [25]:
store = VectorStore()   # first run downloads the embedding model (~130 MB)
ingestor = ConstitutionIngestor(PDFExtractor(), TextCleaner(), ArticleChunker(), store)
chunks = ingestor.run(PDF_PATH, reset=True)
print(len(chunks), "chunks;", store.collection.count(), "stored")   # expect 463 and 463

# semantic search
res = store.query("freedom of speech and the press", n_results=3)
for meta in res["metadatas"][0]:
    print(meta["article_number"], "-", meta.get("article_title"))

# exact lookup + metadata filter
print(store.get_article("19A")["metadatas"][0]["article_title"])      # Right to information.
print(len(store.collection.get(where={"omitted": True})["ids"]))      # 13

/workspaces/Law_bot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6803.00it/s]


463 chunks; 463 stored
19 - Freedom of speech, etc.
66 - Privileges of members, etc.
19A - Right to information.
Right to information.
13


In [28]:
tests = {
    "What is the minimum age to become President?": "41",
    "Who appoints the Chief Justice of Pakistan?": "175A",
    "Can a person be detained without trial?": "10",
    "What does the Constitution say about the Objectives Resolution?": "2A",
    "How can the Constitution be amended?": "239",
}
for q, expected in tests.items():
    res = store.query(q, n_results=5)
    got = [m.get("article_number") or m.get("section") for m in res["metadatas"][0]]
    print(("OK  " if expected in got else "MISS"), q, "->", got)

OK   What is the minimum age to become President? -> ['41', '44', '25A', '62', '175I']
OK   Who appoints the Chief Justice of Pakistan? -> ['177', '175A', '176', '175A', '175A']
OK   Can a person be detained without trial? -> ['10', '10', '10', '10', '203DD']
OK   What does the Constitution say about the Objectives Resolution? -> ['2A', 'Annex (Objectives Resolution)', 'Annex (Objectives Resolution)', '8', '175G']
OK   How can the Constitution be amended? -> ['238', '239', '239', '268', 'Preamble']
